In [1]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

path_transacciones = 'Número de transacciones inmobiliarias de vivienda según residencia del comprador.xls'
# 1. Cargamos el archivo sin saltarnos filas para controlarlo nosotros
df_raw = pd.read_excel(path_transacciones)

# 2. Nos quedamos solo con las filas de datos reales (de la fila 11 en adelante)
# y con las dos columnas que nos interesan (la de la fecha y la del valor)
df_clean = df_raw.iloc[11:, [1, 2]].copy()

# 3. Nombramos las columnas correctamente
df_clean.columns = ['Periodo', 'Total_Transacciones']

# 4. Limpiamos el periodo: Como los años y trimestres vienen mezclados, 
# vamos a propagar el año para que cada fila sepa a qué año pertenece.
año_actual = None
años = []
trimestres = []

for val in df_clean['Periodo']:
    val_str = str(val).strip()
    # Si tiene 4 dígitos es un año (ej. 2007)
    if len(val_str) == 4 and val_str.isdigit():
        año_actual = val_str
        años.append(año_actual)
        trimestres.append('Total Anual')
    else:
        años.append(año_actual)
        trimestres.append(f'T{val_str}')

# Añadimos las columnas estructuradas
df_clean['Año'] = años
df_clean['Trimestre'] = trimestres

# 5. Reorganizamos las columnas para que quede bonito y profesional
df_clean = df_clean[['Año', 'Trimestre', 'Total_Transacciones']]

# 6. Quitamos las filas que sean el "Total Anual" para dejar solo el histórico por trimestres
# (Esto evita duplicar datos si luego sumas en SQL o Power BI)
df_trimestral = df_clean[df_clean['Trimestre'] != 'Total Anual'].copy()

# Resetear el índice
df_trimestral.reset_index(drop=True, inplace=True)

# Ver el resultado
print(df_trimestral.head(8))

# 7. Exportar a CSV
df_trimestral.to_csv('transacciones_cantabria_trimestral.csv', index=False, encoding='utf-8')

¡Estructura final conseguida!
    Año Trimestre Total_Transacciones
0  2007        T1                3360
1  2007        T2                3789
2  2007        T3                3094
3  2007        T4                3163
4  2008        T1                2732
5  2008        T2                2861
6  2008        T3                2210
7  2008        T4                2037


In [8]:
print(df_trimestral.iloc[:15, :4])

     Año Trimestre Total_Transacciones
0   2007        T1                3360
1   2007        T2                3789
2   2007        T3                3094
3   2007        T4                3163
4   2008        T1                2732
5   2008        T2                2861
6   2008        T3                2210
7   2008        T4                2037
8   2009        T1                1908
9   2009        T2                2397
10  2009        T3                2189
11  2009        T4                2311
12  2010        T1                1833
13  2010        T2                2987
14  2010        T3                1173


In [9]:
import os
# Lista de archivos que tienen la estructura temporal en las filas (hacia abajo)
archivos_series_temporales = [


    'Número de transacciones inmobiliarias de vivienda según residencia del comprador.xls',
    'Número de transacciones inmobiliarias de vivienda realizadas por residentes en España según comunidad autónoma del comprador.xls',
    'Estadística de vivienda libre por años.xls',
    'Valor de las transacciones inmobiliarias de vivienda libre.xls'
]

def limpiar_serie_icane(nombre_archivo):
    if not os.path.exists(nombre_archivo):
        print(f"⚠️ Archivo no encontrado: {nombre_archivo}")
        return
    
    print(f"Procesando: {nombre_archivo}...")
    
    # 1. Leer el archivo completo
    df_raw = pd.read_excel(nombre_archivo)
    
    # 2. Identificar dinámicamente dónde empiezan los datos
    # Buscamos la fila donde aparece un año de 4 dígitos en la columna 1 (índice 1)
    fila_inicio = None
    for i, val in enumerate(df_raw.iloc[:, 1]):
        val_str = str(val).strip()
        if len(val_str) == 4 and val_str.isdigit():
            fila_inicio = i
            break
            
    if fila_inicio is None:
        print(f"❌ No se pudo identificar el inicio de los datos en {nombre_archivo}")
        return

    # 3. Extraer columnas clave: Columna 1 (Periodo) y Columna 2 (Datos del indicador)
    # Nota: Si el archivo de Comunidades Autónomas tiene más columnas, las trataremos luego.
    df_clean = df_raw.iloc[fila_inicio:, [1, 2]].copy()
    df_clean.columns = ['Periodo', 'Valor']
    
    # 4. Propagar el año y estructurar Trimestres
    año_actual = None
    ids_año = []
    ids_trimestre = []
    
    for val in df_clean['Periodo']:
        val_str = str(val).strip()
        
        # Caso 1: Es un año completo (4 dígitos)
        if len(val_str) == 4 and val_str.isdigit():
            año_actual = val_str
            ids_año.append(año_actual)
            ids_trimestre.append('Total Anual')
        # Caso 2: Es un trimestre suelto (1, 2, 3, 4 o T1, T2...)
        elif val_str in ['1', '2', '3', '4'] or val_str.startswith('T'):
            ids_año.append(año_actual)
            t = val_str.replace('T', '') # Limpiar por si acaso ya lleva la T
            ids_trimestre.append(f'T{t}')
        # Caso 3: Celdas vacías o totales raros
        else:
            ids_año.append(año_actual)
            ids_trimestre.append('Otros')

    df_clean['Año'] = ids_año
    df_clean['Trimestre'] = ids_trimestre
    
    # 5. Filtrar para quedarnos solo con trimestres puros (quitando totales anuales y vacíos)
    df_final = df_clean[df_clean['Trimestre'].str.startswith('T')].copy()
    
    # Organizar columnas finales
    df_final = df_final[['Año', 'Trimestre', 'Valor']]
    df_final.reset_index(drop=True, inplace=True)
    
    # Generar nombre de salida
    nombre_salida = nombre_archivo.replace('.xls', '_clean.csv')
    df_final.to_csv(nombre_salida, index=False, encoding='utf-8')
    print(f"✅ ¡Guardado con éxito! -> {nombre_salida}\n")

# Ejecutar el limpiador automático para tu lista de archivos
for archivo in archivos_series_temporales:
    limpiar_serie_icane(archivo)

Procesando: Número de transacciones inmobiliarias de vivienda según residencia del comprador.xls...
✅ ¡Guardado con éxito! -> Número de transacciones inmobiliarias de vivienda según residencia del comprador_clean.csv

Procesando: Número de transacciones inmobiliarias de vivienda realizadas por residentes en España según comunidad autónoma del comprador.xls...
✅ ¡Guardado con éxito! -> Número de transacciones inmobiliarias de vivienda realizadas por residentes en España según comunidad autónoma del comprador_clean.csv

Procesando: Estadística de vivienda libre por años.xls...
✅ ¡Guardado con éxito! -> Estadística de vivienda libre por años_clean.csv

Procesando: Valor de las transacciones inmobiliarias de vivienda libre.xls...
✅ ¡Guardado con éxito! -> Valor de las transacciones inmobiliarias de vivienda libre_clean.csv

